# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using SNPE

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Snapdragon Neural Processing Engine (SNPE) SDK**.

Note: This notebook uses the SNPE command-line flow because it is still widely used for DLC-based deployment. For newer projects, Qualcomm’s QAIRT tools provide a more unified interface around conversion, quantization, execution, and profiling.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to SNPE’s Deep Learning Container (DLC) format.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference.
6. **HTP graph compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325).

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries.
import glob
import os
import random
import torch
import uuid

import numpy as np

from libreyolo import LibreYOLO
from PIL import Image

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (resized to 640×640, normalized to `[0, 1]`).
- A **representative calibration dataset** in SNPE’s `.raw` binary format. SNPE uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `preprocess()` helper function used throughout this pipeline.

In [2]:
def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    Args:
        original_image (np.ndarray): Input image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image normalized to [0, 1].
    """

    image = Image.fromarray(original_image).convert("RGB")
    image = image.resize((size, size), Image.BILINEAR)

    image = np.asarray(image).astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
    image = np.expand_dims(image, axis=0)   # CHW -> NCHW

    return image

### Preparing the Calibration Dataset

SNPE’s quantization tool (`snpe-dlc-quantize`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(H, W, C)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **100 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `snpe-dlc-quantize`.

In [3]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 100

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        image = Image.open(filename)
        original_image = np.array(image)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  777M  100  777M    0     0  17.0M      0  0:00:45  0:00:45 --:--:-- 18.0M
Archive:  val2017.zip
   creating: val/val2017/
 extracting: val/val2017/000000212226.jpg  
 extracting: val/val2017/000000231527.jpg  
 extracting: val/val2017/000000578922.jpg  
 extracting: val/val2017/000000062808.jpg  
 extracting: val/val2017/000000119038.jpg  
 extracting: val/val2017/000000114871.jpg  
 extracting: val/val2017/000000463918.jpg  
 extracting: val/val2017/000000365745.jpg  
 extracting: val/val2017/000000320425.jpg  
 extracting: val/val2017/000000481404.jpg  
 extracting: val/val2017/000000314294.jpg  
 extracting: val/val2017/000000335328.jpg  
 extracting: val/val2017/000000513688.jpg  
 extracting: val/val2017/000000158548.jpg  
 extracting: val/val2017/000000132116.jpg  
 extracting: val/val2017/000000415238.jpg  
 extracting

### Loading LibreYOLO and Exporting to ONNX

SNPE does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which SNPE can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=18`.

The model produces **three output heads** corresponding to different detection scales:
- `output_small` (80×80) — detects small objects.
- `output_medium` (40×40) — detects medium objects.
- `output_large` (20×20) — detects large objects.

In [4]:
if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

yolo = LibreYOLO("../models/LibreYOLOXs.pt")
torch_model = yolo.model
torch_model.eval()

if not os.path.exists("../models/LibreYOLOXs.onnx"):
    dummy = torch.randn(1, 3, 640, 640)
    torch.onnx.export(
        torch_model,
        dummy,
        "../models/LibreYOLOXs.onnx",
        opset_version=18,
        dynamo=False,
        input_names=["images"],
        output_names=[
            "output_small",   # 80x80
            "output_medium",  # 40x40
            "output_large"    # 20x20
        ]
    )

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1351  100  1351    0     0   4730      0 --:--:-- --:--:-- --:--:--  4723
100 68.7M  100 68.7M    0     0  29.5M      0  0:00:02  0:00:02 --:--:-- 37.6M


/tmp/ipykernel_3659/3572181382.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


## Converting the Model to SNPE DLC Format

SNPE uses its own model format called the **DLC (Deep Learning Container)**. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `snpe-onnx-to-dlc` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [5]:
!bash -c 'snpe-onnx-to-dlc  \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/snpe/LibreYOLOXs_fp32.dlc'

2026-05-09 13:27:22,044 - 278 - INFO - Input shape info 
2026-05-09 13:27:23,717 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-09 13:27:23,717 - 283 - WARNING - Simplified model validation failed
2026-05-09 13:27:24,068 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-09 13:27:24,068 - 283 - WARNING - Simplified model validation failed
2026-05-09 13:27:24,521 - 278 - INFO - INFO_INITIALIZATION_SUCCESS: 
2026-05-09 13:27:24,912 - 278 - INFO - INFO_CONVERSION_SUCCESS: Conversion completed successfully
2026-05-09 13:27:24,916 - 278 - INFO - INFO_WRITE_SUCCESS: 


### Inspecting the FP32 DLC

The `snpe-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by SNPE before proceeding to quantization.

In [6]:
!bash -c 'snpe-dlc-info  \
    --input_dlc ../models/snpe/LibreYOLOXs_fp32.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_fp32.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; unroll_gru_time_steps=True; quantization_overrides=; prepare_inputs_as_params=False; perform_axes_to_spatial_first_order=True; unroll_lstm_time_steps=True; packed_max_seq=1; validation_target=[]; packed_masked_softmax_inputs=[]; optimization_pass_mode=ir_optimizer_mainline; op_package_lib=; no_simplification=False; multi_time_steps_gru=False; match_caffe_ssd_to_tf=True; preserve_onnx_output_order=False; perform_layout_transformation=False; keep_quant_nodes=False; input_dtype=[]; keep_disconnected_nodes=False; ir_optimizer_config=; input_encoding=[]; handle_gather_negative_indices=True; copyright_file=None; force_prune_cast_ops=False; use_convert_quantization_nodes=False; package_name=None; defer_loading=False; dry_run=None; input_type=[]; float_bitwidth=32; out_

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `snpe-dlc-quantize` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing an INT8 DLC.

In [7]:
!bash -c 'snpe-dlc-quantize  \
    --input_dlc ../models/snpe/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/snpe/LibreYOLOXs_int8.dlc'

/qairt/2.41.0.251128/bin/x86_64-linux-clang/snpe-dlc-quantize: line 224: [: too many arguments
[INFO] InitializeStderr: DebugLog initialized.
[INFO] Processed command-line arguments
     0.7ms [  INFO ] Inferences will run in sync mode
Processing inference input(s):
raw/f4226a75-43c8-40bc-956e-f80daf3b4654.raw
raw/b1e79729-2b54-4a2e-a5a6-8ca7f7470b51.raw
raw/0e040cc5-2a5f-411a-aa4b-7d5c27b57249.raw
raw/d51e7430-77ec-490d-bfa4-511f86581cd7.raw
raw/ca3edc56-d546-4d9e-acbf-57e001d58e30.raw
raw/5f1bbf7b-ae92-4cbf-bd48-5295395c0114.raw
raw/32f19940-805d-4f92-92ff-c39c67ec5d5a.raw
raw/c4ca9e7b-5995-4db8-ab4a-5845c4808209.raw
raw/0854c5f7-f945-414c-870d-47046a1f6611.raw
raw/be0cfc04-1d28-427f-bbde-9e5ba1324cc4.raw
raw/b7e31f9c-a34a-499c-a15d-95497435c1cd.raw
raw/c1616a91-e9c9-4738-b61d-35cb7efbc7e8.raw
raw/bb7bad90-327a-4e96-80ce-1f1d48da13f8.raw
raw/00163764-bdd3-4a6b-a25e-af9c18c2eb39.raw
raw/5eecd45a-3a4a-493d-b1c0-d8f985ef8954.raw
raw/245892c8-68ff-41aa-8777-de9cbc6f9fd7.raw
raw/43f66bb5-

### Inspecting the INT8 DLC

After quantization, `snpe-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [8]:
!bash -c 'snpe-dlc-info  \
    --input_dlc ../models/snpe/LibreYOLOXs_int8.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_int8.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; dump_inferred_model=False; dump_ir_optimizer_config_template=False; input_dim=None; converter_op_package_lib=; adjust_nms_features_dims=True; define_symbol=None; squash_box_decoder=True; inject_cast_for_gather=True; extract_color_transform=True; align_matmul_ranks=True; dumpIR=False; dump_custom_io_config_template=; dump_value_info=False; optimization_pass_mode_config=; preprocess_roi_pool_inputs=True; custom_op_config_paths=None; batch=None; use_quantize_v2=False; keep_int64_inputs=False; float_bias_bw=0; disable_batchnorm_folding=False; enable_strict_validation=False; disable_defer_loading=False; enable_framework_trace=False; dump_qairt_io_config_yaml=; preserve_io=[]; debug=-1; multi_time_steps_lstm=False; expand_lstm_op_structure=False; dump_pass_trace_info=

### Compiling the Graph for the Hexagon Tensor Processor (HTP)

Qualcomm® Snapdragon SoCs include a dedicated AI accelerator called the **Hexagon Tensor Processor (HTP)**. To fully leverage HTP hardware, SNPE can perform an **offline graph compilation** step that optimizes the DLC graph layout for a specific target SoC.

The `snpe-dlc-graph-prepare` command takes the INT8 DLC and the target SoC identifier — `sm7325`, corresponding to the **Snapdragon 778G** — and produces a pre-compiled DLC ready for on-device deployment with maximum HTP utilization.

In [9]:
!bash -c 'snpe-dlc-graph-prepare  \
    --input_dlc ../models/snpe/LibreYOLOXs_int8.dlc  \
    --output_dlc ../models/snpe/LibreYOLOXs_int8_sm7325.dlc \
    --htp_socs sm7325'

[INFO] InitializeStderr: DebugLog initialized.
[INFO] SNPE HTP Offline Prepare: Attempting to create cache for SM7325
[USER_INFO] Target device backend record identifier: HTP_V68_SM7325_2MB
[USER_INFO] No cache record in the DLC matches the target device (HTP_V68_SM7325_2MB). Creating a new record
[USER_INFO] Checking unsigned PD session
[INFO] Attempting to open dynamically linked lib: libHtpPrepare.so
[INFO] dlopen libHtpPrepare.so SUCCESS handle 0x55c82aba1ab0
[INFO] Found Interface Provider (v2.31)
[USER_WARNING]  <W> Initializing HtpProvider
[USER_WARNING]  <W> HTP arch will be deprecated, please set SoC id instead.
[USER_WARNING]  <W> Performance Estimates unsupported on soc SM7325
[USER_WARNING]  <W> Cost Based unsupported on soc SM7325
[USER_INFO] Platform option not set
[USER_INFO] Created ctx=0x1 for Graph Id=0 backend=HTP SNPE Id=0x55c82a9fd7f8
[USER_INFO] Context [0x1] Setting priority to: default
[USER_INFO] Offline Prepare VTCM size(MB) selected = 0
[USER_INFO] Optimizati

### Inspecting the HTP-Optimized DLC

A final call to `snpe-dlc-info` confirms that the DLC has been compiled with HTP-specific optimizations for the **sm7325** SoC and is ready for on-device deployment.

In [10]:
!bash -c 'snpe-dlc-info \
    --input_dlc ../models/snpe/LibreYOLOXs_int8_sm7325.dlc'

DLC info of: /workspace/models/snpe/LibreYOLOXs_int8_sm7325.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: snpe-onnx-to-dlc; dump_inferred_model=False; dump_ir_optimizer_config_template=False; input_dim=None; converter_op_package_lib=; adjust_nms_features_dims=True; define_symbol=None; squash_box_decoder=True; inject_cast_for_gather=True; extract_color_transform=True; align_matmul_ranks=True; dumpIR=False; dump_custom_io_config_template=; dump_value_info=False; optimization_pass_mode_config=; preprocess_roi_pool_inputs=True; custom_op_config_paths=None; batch=None; use_quantize_v2=False; keep_int64_inputs=False; float_bias_bw=0; disable_batchnorm_folding=False; enable_strict_validation=False; disable_defer_loading=False; enable_framework_trace=False; dump_qairt_io_config_yaml=; preserve_io=[]; debug=-1; multi_time_steps_lstm=False; expand_lstm_op_structure=False; dump_pass_trac

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.